![VeritasGraph Banner](../assets/veritasgraph_banner.png)

# 🌳 Hierarchical Tree Extraction - Accuracy Testing

**VeritasGraph v0.2.0** - The Power of PageIndex's Tree + The Flexibility of a Graph

---

This notebook tests the accuracy of VeritasGraph's hierarchical tree extraction feature, which:
- Detects Table of Contents (TOC) in documents
- Extracts parent-child section relationships
- Builds a navigable tree structure inside the knowledge graph

## What We'll Test

| Test | Description |
|------|-------------|
| TOC Detection | Can we correctly identify TOC pages? |
| Structure Extraction | Are section numbers (1, 1.1, 1.1.2) correctly parsed? |
| Parent-Child Relationships | Are parent-child links accurate? |
| Page Range Inference | Do sections span the correct pages? |
| Tree Navigation | Can we navigate the tree accurately? |

## Step 0: Setup

In [ ]:
# Install dependencies if needed
%pip install -q veritasgraph pillow

In [ ]:
import sys
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import json

# Add project root to path for local development
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import VeritasGraph components
from veritasgraph.models import (
    TreeNode,
    HierarchicalStructure,
    SectionType,
    DocumentPage,
)
from veritasgraph.tree_extractor import (
    HierarchicalTreeExtractor,
    TreeQueryEngine,
)

print("✅ VeritasGraph modules loaded successfully!")

## Step 1: Define Ground Truth Test Data

We'll create synthetic test cases with known hierarchical structures to measure accuracy.

In [ ]:
@dataclass
class TestCase:
    """Defines a test case with expected hierarchical structure."""
    name: str
    description: str
    expected_sections: List[Dict]  # [{structure, title, level, parent_structure, start_page}]
    expected_toc_pages: List[int] = field(default_factory=list)
    total_pages: int = 10


# Define test cases with ground truth
TEST_CASES = [
    TestCase(
        name="Simple 2-Level Document",
        description="Basic document with chapters and sections",
        expected_toc_pages=[2],
        total_pages=20,
        expected_sections=[
            {"structure": "1", "title": "Introduction", "level": 1, "parent_structure": "root", "start_page": 3},
            {"structure": "1.1", "title": "Background", "level": 2, "parent_structure": "1", "start_page": 3},
            {"structure": "1.2", "title": "Motivation", "level": 2, "parent_structure": "1", "start_page": 5},
            {"structure": "2", "title": "Methods", "level": 1, "parent_structure": "root", "start_page": 7},
            {"structure": "2.1", "title": "Data Collection", "level": 2, "parent_structure": "2", "start_page": 7},
            {"structure": "2.2", "title": "Analysis", "level": 2, "parent_structure": "2", "start_page": 9},
            {"structure": "3", "title": "Results", "level": 1, "parent_structure": "root", "start_page": 12},
            {"structure": "4", "title": "Conclusion", "level": 1, "parent_structure": "root", "start_page": 18},
        ],
    ),
    TestCase(
        name="Deep 4-Level Hierarchy",
        description="Technical document with deep nesting",
        expected_toc_pages=[2, 3],
        total_pages=50,
        expected_sections=[
            {"structure": "1", "title": "System Architecture", "level": 1, "parent_structure": "root", "start_page": 4},
            {"structure": "1.1", "title": "Overview", "level": 2, "parent_structure": "1", "start_page": 4},
            {"structure": "1.2", "title": "Components", "level": 2, "parent_structure": "1", "start_page": 8},
            {"structure": "1.2.1", "title": "Frontend", "level": 3, "parent_structure": "1.2", "start_page": 8},
            {"structure": "1.2.1.1", "title": "React Components", "level": 4, "parent_structure": "1.2.1", "start_page": 9},
            {"structure": "1.2.1.2", "title": "State Management", "level": 4, "parent_structure": "1.2.1", "start_page": 12},
            {"structure": "1.2.2", "title": "Backend", "level": 3, "parent_structure": "1.2", "start_page": 15},
            {"structure": "1.2.2.1", "title": "API Design", "level": 4, "parent_structure": "1.2.2", "start_page": 16},
            {"structure": "2", "title": "Implementation", "level": 1, "parent_structure": "root", "start_page": 25},
        ],
    ),
    TestCase(
        name="No TOC Document",
        description="Document without explicit TOC - structure extracted from headers",
        expected_toc_pages=[],  # No TOC
        total_pages=15,
        expected_sections=[
            {"structure": "1", "title": "Executive Summary", "level": 1, "parent_structure": "root", "start_page": 1},
            {"structure": "2", "title": "Analysis", "level": 1, "parent_structure": "root", "start_page": 3},
            {"structure": "2.1", "title": "Market Trends", "level": 2, "parent_structure": "2", "start_page": 4},
            {"structure": "2.2", "title": "Competitor Analysis", "level": 2, "parent_structure": "2", "start_page": 7},
            {"structure": "3", "title": "Recommendations", "level": 1, "parent_structure": "root", "start_page": 12},
        ],
    ),
]

print(f"📋 Defined {len(TEST_CASES)} test cases:")
for tc in TEST_CASES:
    print(f"   • {tc.name}: {len(tc.expected_sections)} sections, TOC on pages {tc.expected_toc_pages or 'None'}")

## Step 2: Create Mock Vision Client for Testing

We'll create a mock vision client that returns predictable responses for testing accuracy.

In [ ]:
class MockVisionClient:
    """
    Mock vision client for testing tree extraction accuracy.
    Returns predefined responses based on test case configuration.
    """
    
    def __init__(self, test_case: TestCase):
        self.test_case = test_case
        self._build_response_map()
    
    def _build_response_map(self):
        """Build expected responses for each page."""
        self.toc_pages = set(self.test_case.expected_toc_pages)
        self.sections_by_page = {}
        
        for section in self.test_case.expected_sections:
            page = section["start_page"]
            if page not in self.sections_by_page:
                self.sections_by_page[page] = []
            self.sections_by_page[page].append(section)
    
    def analyze_with_json(self, image, prompt: str) -> Dict:
        """Mock analyze method that returns test data."""
        # Determine page number from image (mock: use hash as page number)
        page_num = getattr(image, '_mock_page_num', 1)
        
        if "toc_detection" in prompt.lower() or "table of contents" in prompt.lower():
            return self._mock_toc_detection(page_num)
        elif "toc_extraction" in prompt.lower() or "extract the complete" in prompt.lower():
            return self._mock_toc_extraction()
        elif "page_structure" in prompt.lower() or "sections and subsections" in prompt.lower():
            return self._mock_page_structure(page_num)
        else:
            return {}
    
    def _mock_toc_detection(self, page_num: int) -> Dict:
        """Return TOC detection result."""
        is_toc = page_num in self.toc_pages
        return {
            "is_toc_page": is_toc,
            "confidence": 0.95 if is_toc else 0.1,
            "toc_type": "explicit" if is_toc else "none",
            "has_page_numbers": is_toc,
            "structure_type": "numbered" if is_toc else "none",
            "reasoning": f"Page {page_num} {'contains' if is_toc else 'does not contain'} TOC"
        }
    
    def _mock_toc_extraction(self) -> Dict:
        """Return TOC extraction result with all sections."""
        entries = [
            {
                "structure": s["structure"],
                "title": s["title"],
                "page": s["start_page"],
                "level": s["level"]
            }
            for s in self.test_case.expected_sections
        ]
        return {
            "toc_entries": entries,
            "total_entries": len(entries),
            "has_page_numbers": True
        }
    
    def _mock_page_structure(self, page_num: int) -> Dict:
        """Return page structure analysis for non-TOC extraction."""
        sections = self.sections_by_page.get(page_num, [])
        return {
            "sections_found": [
                {
                    "title": s["title"],
                    "structure_number": s["structure"],
                    "level": s["level"],
                    "section_type": "chapter" if s["level"] == 1 else "section",
                    "appears_to_start_here": True,
                    "confidence": 0.9
                }
                for s in sections
            ],
            "page_type": "content",
            "has_clear_structure": len(sections) > 0
        }


class MockImage:
    """Mock PIL Image for testing."""
    def __init__(self, page_num: int):
        self._mock_page_num = page_num


print("✅ MockVisionClient created for controlled testing")

## Step 3: Accuracy Measurement Functions

In [ ]:
@dataclass
class AccuracyMetrics:
    """Holds accuracy metrics for tree extraction."""
    test_name: str
    
    # TOC Detection
    toc_detection_correct: bool = False
    toc_pages_precision: float = 0.0
    toc_pages_recall: float = 0.0
    
    # Section Extraction
    sections_extracted: int = 0
    sections_expected: int = 0
    section_precision: float = 0.0  # What % of extracted sections are correct
    section_recall: float = 0.0     # What % of expected sections were extracted
    
    # Structure Accuracy
    structure_accuracy: float = 0.0   # % of sections with correct structure number
    level_accuracy: float = 0.0       # % of sections with correct level
    
    # Parent-Child Relationships
    parent_child_accuracy: float = 0.0  # % of correct parent assignments
    
    # Page Range Accuracy
    page_range_accuracy: float = 0.0   # % of sections with correct start page
    
    # Overall Score
    overall_score: float = 0.0
    
    def calculate_overall(self):
        """Calculate weighted overall score."""
        weights = {
            "section_recall": 0.25,
            "structure_accuracy": 0.20,
            "level_accuracy": 0.15,
            "parent_child_accuracy": 0.25,
            "page_range_accuracy": 0.15,
        }
        self.overall_score = (
            self.section_recall * weights["section_recall"] +
            self.structure_accuracy * weights["structure_accuracy"] +
            self.level_accuracy * weights["level_accuracy"] +
            self.parent_child_accuracy * weights["parent_child_accuracy"] +
            self.page_range_accuracy * weights["page_range_accuracy"]
        )
    
    def __str__(self):
        return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 ACCURACY REPORT: {self.test_name}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 TOC Detection:
   • Correct:   {'✅ Yes' if self.toc_detection_correct else '❌ No'}
   • Precision: {self.toc_pages_precision:.1%}
   • Recall:    {self.toc_pages_recall:.1%}

📑 Section Extraction:
   • Extracted: {self.sections_extracted} / {self.sections_expected} expected
   • Precision: {self.section_precision:.1%}
   • Recall:    {self.section_recall:.1%}

🔢 Structure Accuracy:
   • Structure Numbers: {self.structure_accuracy:.1%}
   • Level Assignment:  {self.level_accuracy:.1%}

🌳 Parent-Child Relationships:
   • Accuracy: {self.parent_child_accuracy:.1%}

📄 Page Range Accuracy:
   • Start Pages: {self.page_range_accuracy:.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 OVERALL SCORE: {self.overall_score:.1%}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


def evaluate_extraction(
    extracted: HierarchicalStructure,
    test_case: TestCase
) -> AccuracyMetrics:
    """Evaluate extraction accuracy against ground truth."""
    metrics = AccuracyMetrics(test_name=test_case.name)
    
    # 1. TOC Detection Accuracy
    expected_toc = set(test_case.expected_toc_pages)
    detected_toc = set(extracted.toc_pages)
    
    metrics.toc_detection_correct = (len(expected_toc) > 0) == extracted.toc_detected
    
    if detected_toc:
        metrics.toc_pages_precision = len(expected_toc & detected_toc) / len(detected_toc)
    else:
        metrics.toc_pages_precision = 1.0 if not expected_toc else 0.0
        
    if expected_toc:
        metrics.toc_pages_recall = len(expected_toc & detected_toc) / len(expected_toc)
    else:
        metrics.toc_pages_recall = 1.0 if not detected_toc else 0.0
    
    # 2. Section Extraction Accuracy
    expected_sections = {s["structure"]: s for s in test_case.expected_sections}
    extracted_sections = {
        n.structure: n for n in extracted.nodes.values()
        if n.id != extracted.root_id
    }
    
    metrics.sections_expected = len(expected_sections)
    metrics.sections_extracted = len(extracted_sections)
    
    # Match sections by structure number
    matched_structures = set(expected_sections.keys()) & set(extracted_sections.keys())
    
    if extracted_sections:
        metrics.section_precision = len(matched_structures) / len(extracted_sections)
    else:
        metrics.section_precision = 0.0
        
    if expected_sections:
        metrics.section_recall = len(matched_structures) / len(expected_sections)
    else:
        metrics.section_recall = 1.0
    
    # 3. Structure and Level Accuracy (for matched sections)
    correct_structures = 0
    correct_levels = 0
    correct_parents = 0
    correct_pages = 0
    
    for structure in matched_structures:
        expected = expected_sections[structure]
        extracted_node = extracted_sections[structure]
        
        # Structure is already matched by key, so count it
        correct_structures += 1
        
        # Check level
        if extracted_node.level == expected["level"]:
            correct_levels += 1
        
        # Check parent
        expected_parent = expected["parent_structure"]
        if extracted_node.parent_id:
            parent_node = extracted.nodes.get(extracted_node.parent_id)
            if parent_node:
                actual_parent = parent_node.structure
                if actual_parent == expected_parent:
                    correct_parents += 1
        elif expected_parent == "root":
            correct_parents += 1
        
        # Check start page
        if extracted_node.start_page == expected["start_page"]:
            correct_pages += 1
    
    if matched_structures:
        metrics.structure_accuracy = correct_structures / len(matched_structures)
        metrics.level_accuracy = correct_levels / len(matched_structures)
        metrics.parent_child_accuracy = correct_parents / len(matched_structures)
        metrics.page_range_accuracy = correct_pages / len(matched_structures)
    
    # Calculate overall score
    metrics.calculate_overall()
    
    return metrics


print("✅ Accuracy measurement functions defined")

## Step 4: Run Tree Extraction Tests

In [ ]:
@dataclass
class AccuracyMetrics:
    """Holds accuracy metrics for tree extraction."""
    test_name: str
    
    # TOC Detection
    toc_detection_correct: bool = False
    toc_pages_precision: float = 0.0
    toc_pages_recall: float = 0.0
    
    # Section Extraction
    sections_extracted: int = 0
    sections_expected: int = 0
    section_precision: float = 0.0  # What % of extracted sections are correct
    section_recall: float = 0.0     # What % of expected sections were extracted
    
    # Structure Accuracy
    structure_accuracy: float = 0.0   # % of sections with correct structure number
    level_accuracy: float = 0.0       # % of sections with correct level
    
    # Parent-Child Relationships
    parent_child_accuracy: float = 0.0  # % of correct parent assignments
    
    # Page Range Accuracy
    page_range_accuracy: float = 0.0   # % of sections with correct start page
    
    # Overall Score
    overall_score: float = 0.0
    
    def calculate_overall(self):
        """Calculate weighted overall score."""
        weights = {
            "section_recall": 0.25,
            "structure_accuracy": 0.20,
            "level_accuracy": 0.15,
            "parent_child_accuracy": 0.25,
            "page_range_accuracy": 0.15,
        }
        self.overall_score = (
            self.section_recall * weights["section_recall"] +
            self.structure_accuracy * weights["structure_accuracy"] +
            self.level_accuracy * weights["level_accuracy"] +
            self.parent_child_accuracy * weights["parent_child_accuracy"] +
            self.page_range_accuracy * weights["page_range_accuracy"]
        )
    
    def __str__(self):
        return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 ACCURACY REPORT: {self.test_name}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 TOC Detection:
   • Correct:   {'✅ Yes' if self.toc_detection_correct else '❌ No'}
   • Precision: {self.toc_pages_precision:.1%}
   • Recall:    {self.toc_pages_recall:.1%}

📑 Section Extraction:
   • Extracted: {self.sections_extracted} / {self.sections_expected} expected
   • Precision: {self.section_precision:.1%}
   • Recall:    {self.section_recall:.1%}

🔢 Structure Accuracy:
   • Structure Numbers: {self.structure_accuracy:.1%}
   • Level Assignment:  {self.level_accuracy:.1%}

🌳 Parent-Child Relationships:
   • Accuracy: {self.parent_child_accuracy:.1%}

📄 Page Range Accuracy:
   • Start Pages: {self.page_range_accuracy:.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 OVERALL SCORE: {self.overall_score:.1%}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


def evaluate_extraction(
    extracted: HierarchicalStructure,
    test_case: TestCase
) -> AccuracyMetrics:
    """Evaluate extraction accuracy against ground truth."""
    metrics = AccuracyMetrics(test_name=test_case.name)
    
    # 1. TOC Detection Accuracy
    expected_toc = set(test_case.expected_toc_pages)
    detected_toc = set(extracted.toc_pages)
    
    metrics.toc_detection_correct = (len(expected_toc) > 0) == extracted.toc_detected
    
    if detected_toc:
        metrics.toc_pages_precision = len(expected_toc & detected_toc) / len(detected_toc)
    else:
        metrics.toc_pages_precision = 1.0 if not expected_toc else 0.0
        
    if expected_toc:
        metrics.toc_pages_recall = len(expected_toc & detected_toc) / len(expected_toc)
    else:
        metrics.toc_pages_recall = 1.0 if not detected_toc else 0.0
    
    # 2. Section Extraction Accuracy
    expected_sections = {s["structure"]: s for s in test_case.expected_sections}
    extracted_sections = {
        n.structure: n for n in extracted.nodes.values()
        if n.id != extracted.root_id
    }
    
    metrics.sections_expected = len(expected_sections)
    metrics.sections_extracted = len(extracted_sections)
    
    # Match sections by structure number
    matched_structures = set(expected_sections.keys()) & set(extracted_sections.keys())
    
    if extracted_sections:
        metrics.section_precision = len(matched_structures) / len(extracted_sections)
    else:
        metrics.section_precision = 0.0
        
    if expected_sections:
        metrics.section_recall = len(matched_structures) / len(expected_sections)
    else:
        metrics.section_recall = 1.0
    
    # 3. Structure and Level Accuracy (for matched sections)
    correct_structures = 0
    correct_levels = 0
    correct_parents = 0
    correct_pages = 0
    
    for structure in matched_structures:
        expected = expected_sections[structure]
        extracted_node = extracted_sections[structure]
        
        # Structure is already matched by key, so count it
        correct_structures += 1
        
        # Check level
        if extracted_node.level == expected["level"]:
            correct_levels += 1
        
        # Check parent
        expected_parent = expected["parent_structure"]
        if extracted_node.parent_id:
            parent_node = extracted.nodes.get(extracted_node.parent_id)
            if parent_node:
                actual_parent = parent_node.structure
                if actual_parent == expected_parent:
                    correct_parents += 1
        elif expected_parent == "root":
            correct_parents += 1
        
        # Check start page
        if extracted_node.start_page == expected["start_page"]:
            correct_pages += 1
    
    if matched_structures:
        metrics.structure_accuracy = correct_structures / len(matched_structures)
        metrics.level_accuracy = correct_levels / len(matched_structures)
        metrics.parent_child_accuracy = correct_parents / len(matched_structures)
        metrics.page_range_accuracy = correct_pages / len(matched_structures)
    
    # Calculate overall score
    metrics.calculate_overall()
    
    return metrics


print("✅ Accuracy measurement functions defined")

In [ ]:
# Run all tests
all_results = []

for test_case in TEST_CASES:
    hierarchy, metrics = run_extraction_test(test_case)
    all_results.append((test_case, hierarchy, metrics))
    print(metrics)

## Step 5: Visualize Tree Structure

In [ ]:
def visualize_tree_comparison(test_case: TestCase, hierarchy: HierarchicalStructure):
    """Visualize expected vs extracted tree structure."""
    print(f"\n{'='*70}")
    print(f"🌳 Tree Comparison: {test_case.name}")
    print(f"{'='*70}")
    
    # Expected tree
    print("\n📋 EXPECTED STRUCTURE:")
    print("-" * 40)
    for section in test_case.expected_sections:
        indent = "    " * (section["level"] - 1)
        print(f"{indent}[{section['structure']}] {section['title']} (p.{section['start_page']})")
    
    # Extracted tree
    print("\n🔍 EXTRACTED STRUCTURE:")
    print("-" * 40)
    
    query_engine = TreeQueryEngine(hierarchy)
    tree_view = query_engine.get_tree_view(max_depth=5)
    print(tree_view)
    
    # Detailed comparison
    print("\n📊 DETAILED COMPARISON:")
    print("-" * 40)
    print(f"{'Structure':<12} {'Expected':<25} {'Extracted':<25} {'Match'}")
    print("-" * 70)
    
    expected_map = {s["structure"]: s for s in test_case.expected_sections}
    extracted_map = {n.structure: n for n in hierarchy.nodes.values() if n.id != hierarchy.root_id}
    
    all_structures = sorted(set(expected_map.keys()) | set(extracted_map.keys()))
    
    for struct in all_structures:
        exp = expected_map.get(struct)
        ext = extracted_map.get(struct)
        
        exp_title = exp["title"][:22] if exp else "(missing)"
        ext_title = ext.title[:22] if ext else "(missing)"
        
        match = "✅" if exp and ext and exp["title"] == ext.title else "❌"
        print(f"{struct:<12} {exp_title:<25} {ext_title:<25} {match}")


# Visualize each test result
for test_case, hierarchy, metrics in all_results:
    visualize_tree_comparison(test_case, hierarchy)

## Step 6: Aggregate Results Summary

In [ ]:
print("\n" + "="*70)
print("📈 AGGREGATE ACCURACY SUMMARY")
print("="*70)

# Create summary table
print(f"\n{'Test Case':<30} {'Sections':<12} {'Structure':<12} {'Parents':<12} {'Overall':<10}")
print("-"*76)

total_overall = 0
for test_case, hierarchy, metrics in all_results:
    print(
        f"{test_case.name:<30} "
        f"{metrics.section_recall:>10.1%}  "
        f"{metrics.structure_accuracy:>10.1%}  "
        f"{metrics.parent_child_accuracy:>10.1%}  "
        f"{metrics.overall_score:>8.1%}"
    )
    total_overall += metrics.overall_score

print("-"*76)
avg_score = total_overall / len(all_results) if all_results else 0
print(f"{'AVERAGE':<30} {'':<12} {'':<12} {'':<12} {avg_score:>8.1%}")

print("\n" + "="*70)
if avg_score >= 0.9:
    print("🎉 EXCELLENT! Tree extraction accuracy is above 90%")
elif avg_score >= 0.75:
    print("✅ GOOD! Tree extraction accuracy is acceptable (75-90%)")
elif avg_score >= 0.5:
    print("⚠️  NEEDS IMPROVEMENT: Accuracy is between 50-75%")
else:
    print("❌ POOR: Accuracy is below 50%, significant improvements needed")
print("="*70)

## Step 7: Test with Real PDF (Optional)

If you have a PDF with a known structure, you can test accuracy against it.

In [ ]:
# Optional: Test with a real PDF
# Uncomment and modify the path to test with your own document

# from veritasgraph import VisionRAGPipeline
# 
# # Initialize pipeline with tree extraction
# pipeline = VisionRAGPipeline(
#     model_name="llama3.2-vision:11b",
#     extract_tree=True
# )
# 
# # Ingest a PDF
# pdf_path = "../path/to/your/document.pdf"
# doc = pipeline.ingest_pdf(pdf_path)
# 
# # View the extracted tree
# print("\n📑 Extracted Document Tree:")
# print(pipeline.get_document_tree(doc.document_id))
# 
# # Navigate to a section
# result = pipeline.navigate_to_section(doc.document_id, "Introduction")
# if result:
#     print(f"\n📍 Found section: {result['section'].title}")
#     print(f"   Pages: {result['page_range']}")
#     print(f"   Breadcrumb: {' > '.join(n.title for n in result['breadcrumb'])}")

## Step 8: Export Test Results

In [ ]:
# Export results to JSON for tracking over time
import json
from datetime import datetime

results_export = {
    "timestamp": datetime.now().isoformat(),
    "veritasgraph_version": "0.2.0",
    "test_results": [
        {
            "test_name": metrics.test_name,
            "toc_detection_correct": metrics.toc_detection_correct,
            "section_precision": metrics.section_precision,
            "section_recall": metrics.section_recall,
            "structure_accuracy": metrics.structure_accuracy,
            "level_accuracy": metrics.level_accuracy,
            "parent_child_accuracy": metrics.parent_child_accuracy,
            "page_range_accuracy": metrics.page_range_accuracy,
            "overall_score": metrics.overall_score,
        }
        for _, _, metrics in all_results
    ],
    "average_overall_score": avg_score,
}

# Save to file
results_path = Path("tree_extraction_accuracy_results.json")
with open(results_path, "w") as f:
    json.dump(results_export, f, indent=2)

print(f"✅ Results exported to: {results_path.absolute()}")
print("\n📋 Results Preview:")
print(json.dumps(results_export, indent=2))

## 🔬 Step 9: Test with Real PDF using Vision Model

Now let's test with a real PDF document (`chart_demo.pdf`) using the actual vision model.

In [10]:
# Test with real PDF: chart_demo.pdf
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import VeritasGraph Vision Pipeline
from veritasgraph.vision import VisionRAGPipeline, VisionRAGConfig

# PDF path
pdf_path = Path("chart_demo.pdf")

if pdf_path.exists():
    print(f"✅ Found PDF: {pdf_path.absolute()}")
    print(f"   Size: {pdf_path.stat().st_size / 1024:.1f} KB")
else:
    print(f"❌ PDF not found: {pdf_path}")
    print("   Please place chart_demo.pdf in the cookbook directory")

✅ Found PDF: c:\Projects\graphrag\VeritasGraph\cookbook\chart_demo.pdf
   Size: 68.6 KB


In [11]:
# Initialize the Vision RAG Pipeline with tree extraction enabled
config = VisionRAGConfig(
    vision_model="llama3.2-vision:11b",  # Use vision model
    pdf_dpi=150,  # Balance quality vs speed
    output_dir="./vision_output",
    cache_dir="./vision_cache"
)

# Create pipeline with hierarchical tree extraction
pipeline = VisionRAGPipeline(config=config, extract_tree=True)

print("✅ VisionRAGPipeline initialized")
print(f"   Vision Model: {config.vision_model}")
print(f"   Tree Extraction: Enabled")

📋 Available models: ['llama3.2-vision:11b', 'llava:latest', 'qwen3:8b', 'qwen3:latest', 'llama3.1-12k:latest', 'nomic-embed-text:latest', 'deepseek-r1:7b', 'llama3.1:latest', 'llama3:latest']
✅ Vision model ready: llama3.2-vision:11b
✅ VisionRAGPipeline initialized
   Vision Model: llama3.2-vision:11b
   Tree Extraction: Enabled


In [12]:
%%time
# Ingest the PDF - this will extract hierarchical tree structure using vision model
print("🚀 Starting PDF ingestion with tree extraction...")
print("   This may take a few minutes depending on document size.\n")

doc = pipeline.ingest_pdf(str(pdf_path))

if doc:
    print(f"\n✅ Document ingested: {doc.id}")
    print(f"   Title: {doc.title}")
    print(f"   Pages: {len(doc.pages)}")
else:
    print("❌ Failed to ingest document")

🚀 Starting PDF ingestion with tree extraction...
   This may take a few minutes depending on document size.


📥 INGESTING: chart_demo.pdf
📄 Converting PDF: chart_demo.pdf
   ✅ Converted 1 pages

🌳 Extracting hierarchical tree structure...

🌳 Extracting hierarchical structure for document 3a723518b5e4...
   📄 No TOC detected, analyzing page structure...
   ✅ Extracted 9 tree nodes
   📊 Max depth: 5
   ✅ Tree structure extracted: 9 sections

🔬 Analyzing 1 pages...
   📖 Analyzing page 1...
      📊 Extracting tables...
      📈 Extracting charts...
      🔢 Extracting metrics...
      🏷️ Extracting entities...
      📝 Generating summary...

🔗 Building knowledge graph for: chart_demo.pdf
   🌳 Adding hierarchical tree structure...
   ✅ Graph built: 18 nodes, 45 edges
   🌳 Tree structure: 9 sections

✅ Document ingested successfully!
   📄 Pages: 1
   🌳 Sections: 9
   📊 Tree depth: 5 levels
   📊 Tables: 3
   📈 Charts: 1
   🔢 Metrics: 3
   🏷️ Entities: 0

✅ Document ingested: 3a723518b5e4
   Titl

In [13]:
# Display the extracted hierarchical tree structure
print("🌳 EXTRACTED DOCUMENT TREE STRUCTURE")
print("="*60)

if doc and doc.hierarchy:
    # Get tree visualization
    tree_view = pipeline.get_document_tree(doc.id)
    print(tree_view)
    
    # Print statistics
    print("\n📊 TREE STATISTICS")
    print("-"*40)
    print(f"   Total Sections: {len(doc.hierarchy.nodes)}")
    print(f"   TOC Detected: {'Yes' if doc.hierarchy.toc_detected else 'No'}")
    if doc.hierarchy.toc_pages:
        print(f"   TOC Pages: {doc.hierarchy.toc_pages}")
    
    # Level distribution
    level_counts = {}
    for node in doc.hierarchy.nodes.values():
        level_counts[node.level] = level_counts.get(node.level, 0) + 1
    
    print(f"\n   Sections by Level:")
    for level in sorted(level_counts.keys()):
        print(f"      Level {level}: {level_counts[level]} sections")
else:
    print("❌ No hierarchy extracted")

🌳 EXTRACTED DOCUMENT TREE STRUCTURE
Document Root (p. 1)
    [1] Quarterly Revenue Report (p. 1)
        [1.1] 2024 Quarterly Revenue Performance (p. 1)
            [1.1.1] Detailed Breakdown (p. 1)
                [1.1.1.1] Quarter (p. 1)
                │   [1.1.1.1.1] Q1 2024 (p. 1)
                │   [1.1.1.1.2] Q2 2024 (p. 1)
                │   [1.1.1.1.3] Q3 2024 (p. 1)
                    [1.1.1.1.4] Q4 2024 (p. 1)

📊 TREE STATISTICS
----------------------------------------
   Total Sections: 9
   TOC Detected: No

   Sections by Level:
      Level 0: 1 sections
      Level 1: 1 sections
      Level 2: 1 sections
      Level 3: 1 sections
      Level 4: 1 sections
      Level 5: 4 sections


In [14]:
# Examine extracted sections in detail
print("📑 DETAILED SECTION ANALYSIS")
print("="*70)

if doc and doc.hierarchy:
    print(f"\n{'#':<4} {'Structure':<12} {'Level':<6} {'Type':<15} {'Title':<30} {'Pages'}")
    print("-"*85)
    
    # Sort nodes by structure for readable output
    sorted_nodes = sorted(
        [n for n in doc.hierarchy.nodes.values() if n.id != doc.hierarchy.root_id],
        key=lambda x: (
            tuple(int(p) if p.isdigit() else ord(p[0]) 
                  for p in x.structure.split('.')) 
            if x.structure and x.structure != 'root' else (0,)
        )
    )
    
    for i, node in enumerate(sorted_nodes, 1):
        page_range = ""
        if node.start_page:
            if node.end_page and node.end_page != node.start_page:
                page_range = f"p.{node.start_page}-{node.end_page}"
            else:
                page_range = f"p.{node.start_page}"
        
        title_short = node.title[:28] + ".." if len(node.title) > 30 else node.title
        print(f"{i:<4} {node.structure:<12} {node.level:<6} {node.section_type.value:<15} {title_short:<30} {page_range}")
else:
    print("❌ No hierarchy to analyze")

📑 DETAILED SECTION ANALYSIS

#    Structure    Level  Type            Title                          Pages
-------------------------------------------------------------------------------------
1    1            1      chapter         Quarterly Revenue Report       p.1
2    1.1          2      section         2024 Quarterly Revenue Perfo.. p.1
3    1.1.1        3      subsection      Detailed Breakdown             p.1
4    1.1.1.1      4      subsubsection   Quarter                        p.1
5    1.1.1.1.1    5      paragraph       Q1 2024                        p.1
6    1.1.1.1.2    5      paragraph       Q2 2024                        p.1
7    1.1.1.1.3    5      paragraph       Q3 2024                        p.1
8    1.1.1.1.4    5      paragraph       Q4 2024                        p.1


In [15]:
# Test tree navigation - find a section by title
print("🧭 TREE NAVIGATION TEST")
print("="*60)

if doc and doc.hierarchy:
    # Test searching for sections
    from veritasgraph.tree_extractor import TreeQueryEngine
    
    query_engine = TreeQueryEngine(doc.hierarchy)
    
    # Try to find sections (adjust these based on your document)
    test_queries = ["introduction", "data", "results", "conclusion", "chart", "table"]
    
    print("\n🔍 Section Search Results:")
    print("-"*50)
    
    for query in test_queries:
        matches = query_engine.find_sections_by_title(query, fuzzy=True)
        if matches:
            print(f"\n   Query: '{query}'")
            for match in matches[:3]:  # Show top 3 matches
                print(f"      → [{match.structure}] {match.title}")
        else:
            print(f"\n   Query: '{query}' - No matches found")
else:
    print("❌ No hierarchy for navigation test")

🧭 TREE NAVIGATION TEST

🔍 Section Search Results:
--------------------------------------------------

   Query: 'introduction' - No matches found

   Query: 'data' - No matches found

   Query: 'results' - No matches found

   Query: 'conclusion' - No matches found

   Query: 'chart' - No matches found

   Query: 'table' - No matches found


In [16]:
# Validate parent-child relationships
print("🔗 PARENT-CHILD RELATIONSHIP VALIDATION")
print("="*60)

if doc and doc.hierarchy:
    errors = []
    validated = 0
    
    for node in doc.hierarchy.nodes.values():
        if node.id == doc.hierarchy.root_id:
            continue
            
        validated += 1
        
        # Check parent exists
        if node.parent_id:
            parent = doc.hierarchy.nodes.get(node.parent_id)
            if not parent:
                errors.append(f"❌ {node.title}: Parent '{node.parent_id}' not found")
            elif node.id not in parent.children_ids:
                errors.append(f"⚠️  {node.title}: Not in parent's children list")
        
        # Check children exist
        for child_id in node.children_ids:
            child = doc.hierarchy.nodes.get(child_id)
            if not child:
                errors.append(f"❌ {node.title}: Child '{child_id}' not found")
            elif child.parent_id != node.id:
                errors.append(f"⚠️  {node.title}: Child '{child.title}' has wrong parent")
    
    # Report results
    if errors:
        print(f"\n⚠️  Found {len(errors)} issues in {validated} relationships:")
        for err in errors[:10]:
            print(f"   {err}")
        if len(errors) > 10:
            print(f"   ... and {len(errors) - 10} more")
    else:
        print(f"\n✅ All {validated} parent-child relationships are valid!")
    
    # Calculate accuracy
    relationship_accuracy = (validated - len(errors)) / validated if validated > 0 else 0
    print(f"\n📊 Relationship Accuracy: {relationship_accuracy:.1%}")
else:
    print("❌ No hierarchy for validation")

🔗 PARENT-CHILD RELATIONSHIP VALIDATION

✅ All 8 parent-child relationships are valid!

📊 Relationship Accuracy: 100.0%


In [17]:
# Comprehensive accuracy report for real PDF
print("="*70)
print("📊 COMPREHENSIVE ACCURACY REPORT - Real PDF Test")
print("="*70)

if doc and doc.hierarchy:
    report = {
        "document": {
            "id": doc.id,
            "title": doc.title,
            "pages": len(doc.pages),
        },
        "tree_extraction": {
            "toc_detected": doc.hierarchy.toc_detected,
            "toc_pages": doc.hierarchy.toc_pages,
            "total_sections": len(doc.hierarchy.nodes) - 1,  # Exclude root
        },
        "metrics": {}
    }
    
    # 1. Structure Coverage
    sections_with_structure = sum(
        1 for n in doc.hierarchy.nodes.values() 
        if n.structure and n.structure != "root" and n.id != doc.hierarchy.root_id
    )
    total_sections = len(doc.hierarchy.nodes) - 1
    report["metrics"]["structure_coverage"] = sections_with_structure / total_sections if total_sections > 0 else 0
    
    # 2. Level Distribution Quality
    levels = [n.level for n in doc.hierarchy.nodes.values() if n.id != doc.hierarchy.root_id]
    max_level = max(levels) if levels else 0
    report["metrics"]["max_depth"] = max_level
    report["metrics"]["avg_depth"] = sum(levels) / len(levels) if levels else 0
    
    # 3. Page Assignment Coverage
    sections_with_pages = sum(
        1 for n in doc.hierarchy.nodes.values() 
        if n.start_page is not None and n.id != doc.hierarchy.root_id
    )
    report["metrics"]["page_coverage"] = sections_with_pages / total_sections if total_sections > 0 else 0
    
    # 4. Parent-Child Integrity
    valid_relationships = 0
    total_relationships = 0
    for node in doc.hierarchy.nodes.values():
        if node.id == doc.hierarchy.root_id:
            continue
        total_relationships += 1
        if node.parent_id and node.parent_id in doc.hierarchy.nodes:
            parent = doc.hierarchy.nodes[node.parent_id]
            if node.id in parent.children_ids:
                valid_relationships += 1
    
    report["metrics"]["relationship_integrity"] = valid_relationships / total_relationships if total_relationships > 0 else 0
    
    # 5. Title Extraction Quality (non-empty, reasonable length)
    good_titles = sum(
        1 for n in doc.hierarchy.nodes.values()
        if n.title and len(n.title) > 2 and len(n.title) < 200 and n.id != doc.hierarchy.root_id
    )
    report["metrics"]["title_quality"] = good_titles / total_sections if total_sections > 0 else 0
    
    # Calculate overall score
    weights = {
        "structure_coverage": 0.20,
        "page_coverage": 0.20,
        "relationship_integrity": 0.30,
        "title_quality": 0.30,
    }
    
    overall = sum(
        report["metrics"].get(k, 0) * v 
        for k, v in weights.items()
    )
    report["metrics"]["overall_score"] = overall
    
    # Print report
    print(f"\n📄 Document: {doc.title}")
    print(f"   Pages: {len(doc.pages)}")
    print(f"   Sections Extracted: {total_sections}")
    print(f"   TOC Detected: {'Yes' if doc.hierarchy.toc_detected else 'No'}")
    
    print(f"\n📈 METRICS:")
    print(f"   Structure Coverage:      {report['metrics']['structure_coverage']:.1%}")
    print(f"   Page Assignment:         {report['metrics']['page_coverage']:.1%}")
    print(f"   Relationship Integrity:  {report['metrics']['relationship_integrity']:.1%}")
    print(f"   Title Quality:           {report['metrics']['title_quality']:.1%}")
    print(f"   Max Tree Depth:          {report['metrics']['max_depth']} levels")
    
    print(f"\n{'='*50}")
    print(f"🎯 OVERALL ACCURACY SCORE: {overall:.1%}")
    print(f"{'='*50}")
    
    if overall >= 0.9:
        print("🎉 EXCELLENT! Tree extraction is highly accurate")
    elif overall >= 0.75:
        print("✅ GOOD! Tree extraction is acceptable")
    elif overall >= 0.5:
        print("⚠️  MODERATE: Some improvements needed")
    else:
        print("❌ NEEDS WORK: Significant improvements required")
        
    # Save report
    import json
    report_path = Path("real_pdf_accuracy_report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"\n💾 Report saved to: {report_path}")
else:
    print("❌ No document or hierarchy available for report")

📊 COMPREHENSIVE ACCURACY REPORT - Real PDF Test

📄 Document: chart_demo.pdf
   Pages: 1
   Sections Extracted: 8
   TOC Detected: No

📈 METRICS:
   Structure Coverage:      100.0%
   Page Assignment:         100.0%
   Relationship Integrity:  100.0%
   Title Quality:           100.0%
   Max Tree Depth:          5 levels

🎯 OVERALL ACCURACY SCORE: 100.0%
🎉 EXCELLENT! Tree extraction is highly accurate

💾 Report saved to: real_pdf_accuracy_report.json


In [18]:
# Export all extracted data to JSON
import os
from pathlib import Path

output_dir = Path("vision_output")
output_dir.mkdir(exist_ok=True)

# 1. Export complete extraction data using pipeline's export method
json_output_path = output_dir / "extracted_data.json"
pipeline.export_to_json(str(json_output_path))

# 2. Export tree structure separately
if doc and doc.hierarchy:
    tree_output_path = output_dir / "tree_structure.json"
    tree_data = doc.hierarchy.to_dict()
    with open(tree_output_path, "w") as f:
        json.dump(tree_data, f, indent=2, default=str)
    print(f"✅ Tree structure exported to: {tree_output_path}")

# 3. Export page-by-page extraction details
if doc:
    pages_output_path = output_dir / "page_extractions.json"
    pages_data = {
        "document_id": doc.id,
        "document_title": doc.title,
        "total_pages": len(doc.pages),
        "pages": []
    }
    
    for page in doc.pages:
        page_info = {
            "page_number": page.page_number,
            "page_type": page.page_type,
            "section_id": page.section_id,
            "summary": page.page_summary,
            "elements": [
                {
                    "id": elem.id,
                    "type": elem.element_type,
                    "description": elem.description,
                    "confidence": elem.confidence,
                    "structured_data": elem.structured_data,
                    "raw_text": elem.raw_text,
                    "section_id": elem.section_id
                }
                for elem in page.elements
            ]
        }
        pages_data["pages"].append(page_info)
    
    with open(pages_output_path, "w") as f:
        json.dump(pages_data, f, indent=2, default=str)
    print(f"✅ Page extractions exported to: {pages_output_path}")

# 4. Export entities
if doc and doc.extracted_entities:
    entities_output_path = output_dir / "extracted_entities.json"
    with open(entities_output_path, "w") as f:
        json.dump(doc.extracted_entities, f, indent=2, default=str)
    print(f"✅ Entities exported to: {entities_output_path}")
else:
    print("ℹ️  No entities were extracted")

# 5. Export knowledge graph data
graph_output_path = output_dir / "knowledge_graph.json"
graph_data = {
    "nodes": [],
    "edges": []
}

for node_id, node_data in pipeline.knowledge_graph.graph.nodes(data=True):
    graph_data["nodes"].append({
        "id": node_id,
        **{k: v for k, v in node_data.items() if not callable(v)}
    })

for source, target, edge_data in pipeline.knowledge_graph.graph.edges(data=True):
    graph_data["edges"].append({
        "source": source,
        "target": target,
        **{k: v for k, v in edge_data.items() if not callable(v)}
    })

with open(graph_output_path, "w") as f:
    json.dump(graph_data, f, indent=2, default=str)
print(f"✅ Knowledge graph exported to: {graph_output_path}")

# Summary
print(f"\n📁 All exports saved to: {output_dir.absolute()}")
print(f"   Files created:")
for f in output_dir.glob("*.json"):
    size_kb = f.stat().st_size / 1024
    print(f"   - {f.name} ({size_kb:.1f} KB)")

✅ Exported to vision_output\extracted_data.json
✅ Tree structure exported to: vision_output\tree_structure.json
✅ Page extractions exported to: vision_output\page_extractions.json
ℹ️  No entities were extracted
✅ Knowledge graph exported to: vision_output\knowledge_graph.json

📁 All exports saved to: c:\Projects\graphrag\VeritasGraph\cookbook\vision_output
   Files created:
   - extracted_data.json (13.9 KB)
   - knowledge_graph.json (17.4 KB)
   - page_extractions.json (6.4 KB)
   - tree_structure.json (3.8 KB)


## 📊 Accuracy Metrics Explained

| Metric | What It Measures | Good Score |
|--------|------------------|------------|
| **Structure Coverage** | % of sections with valid hierarchical numbers (1.1, 1.2, etc.) | >90% |
| **Page Assignment** | % of sections with start page identified | >85% |
| **Relationship Integrity** | % of valid parent-child links | >95% |
| **Title Quality** | % of sections with meaningful titles | >90% |

### How to Improve Accuracy

1. **If TOC not detected**: Ensure document has clear "Table of Contents" or "Contents" heading
2. **If relationships are wrong**: Check that section numbers follow standard format (1, 1.1, 1.1.1)
3. **If pages are missing**: Verify PDF has clear section headers on content pages
4. **If titles are poor**: Ensure section headers are visually distinct from body text

---

## 📚 Summary

This notebook tested the accuracy of VeritasGraph's hierarchical tree extraction:

| Metric | Description | Target |
|--------|-------------|--------|
| **Section Recall** | % of expected sections found | >90% |
| **Structure Accuracy** | Correct hierarchical numbers | >95% |
| **Parent-Child Accuracy** | Correct relationships | >90% |
| **Page Range Accuracy** | Correct page assignments | >85% |

### Next Steps

1. **Add more test cases** - Cover edge cases like appendices, references, multi-TOC documents
2. **Test with real PDFs** - Validate against actual documents with known structure
3. **Tune extraction prompts** - Improve vision model prompts for better accuracy
4. **Add regression tests** - Track accuracy over time as code changes

---

**VeritasGraph v0.2.0** | The Power of PageIndex's Tree + The Flexibility of a Graph